In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# parameters
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

prerefit_start, prerefit_end = '2018-01-01', '2024-04-26'
refit_start,    refit_end    = '2024-04-29', '2026-09-01'

schema = 'xlab_ecb_prj_sftds_cb_common'

In [ ]:
# gov_fut.csv as an inline SQL table, so the CREATE TABLE statements can join to it
gov_fut_sql = '\n    UNION ALL '.join(
    f"SELECT '{r.isin}' AS isin, '{r.country}' AS country, '{r.expiration_date}' AS expiration_date"
    for r in euro_govs.itertuples()
)

In [ ]:
# pre-REFIT EA query (Eurex sovereign futures)
def prerefit_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_counterparty_side AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code AS rep_country,
        e.leg1_other_cpty_id_country_code     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        e.leg1_product_id  AS product_id,
        g.country          AS product_country,
        g.expiration_date  AS expiration_date,
        -- trade terms
        e.leg1_notional            AS notional,
        e.leg1_notional_currency1  AS notional_currency,
        e.leg1_price_rate          AS price,
        e.leg1_quantity            AS quantity
    FROM crp_emir_ecb.emir_ecb_states_deduplicated e
    JOIN (
        {gov_fut_sql}
    ) g
        ON e.leg1_product_id = g.isin
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_counterparty_side IN ('BYER', 'SLLR')
        AND e.leg1_notional > 0
        AND e.leg1_notional <= 1e11
) base
"""

In [ ]:
# REFIT EA query (Eurex sovereign futures)
def refit_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        e.leg1_product_isin  AS product_id,
        g.country            AS product_country,
        g.expiration_date    AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1           AS notional,
        e.leg1_notional_leg1_currency  AS notional_currency,
        e.leg1_price                   AS price,
        e.leg1_notional_quantity_leg1  AS quantity
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    JOIN (
        {gov_fut_sql}
    ) g
        ON e.leg1_product_isin = g.isin
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
        AND e.leg1_notional_leg1 > 0
        AND e.leg1_notional_leg1 <= 1e11
) base
"""

In [ ]:
# pre-REFIT US query (CBOT Treasury futures)
# no contract ISIN on CBOT, so product_id is synthetic: venue, CFI and expiry
def prerefit_us_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_counterparty_side AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code AS rep_country,
        e.leg1_other_cpty_id_country_code     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CONCAT('XCBT_FFDPSX_', to_date(e.leg1_maturity_date)) AS product_id,
        'US'                                                 AS product_country,
        to_date(e.leg1_maturity_date)                        AS expiration_date,
        -- trade terms
        e.leg1_notional            AS notional,
        e.leg1_notional_currency1  AS notional_currency,
        e.leg1_price_rate          AS price,
        e.leg1_quantity            AS quantity
    FROM crp_emir_ecb.emir_ecb_states_deduplicated e
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_execution_venue = 'XCBT'
        AND e.leg1_contract_type = 'FUTR'
        AND e.leg1_asset_class = 'INTR'
        AND e.leg1_product_clssfctn = 'FFDPSX'
        AND e.leg1_notional_currency1 = 'USD'
        AND e.leg1_maturity_date IS NOT NULL
        AND e.leg1_counterparty_side IN ('BYER', 'SLLR')
        AND e.leg1_notional > 0
        AND e.leg1_notional <= 1e11
) base
"""

In [ ]:
# REFIT US query (CBOT Treasury futures)
# no contract ISIN on CBOT, so product_id is synthetic: venue, CFI and expiry
def refit_us_query(start, end):
    return f"""
SELECT
    -- identifiers
    tec_ruti,
    reference_period,
    -- counterparties
    CASE WHEN direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN direction = 'BYER' THEN rep_country            ELSE oth_country            END AS buyer_country,
    CASE WHEN direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector,
    CASE WHEN direction = 'SLLR' THEN rep_country            ELSE oth_country            END AS seller_country,
    -- product
    product_id,
    product_country,
    expiration_date,
    -- trade terms
    notional,
    notional_currency,
    price,
    quantity
FROM (
    SELECT
        -- identifiers
        e.tec_ruti,
        e.reference_period,
        -- counterparties
        e.leg1_direction AS direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        e.leg1_reporting_cpty_id_country_code_gleif AS rep_country,
        e.leg1_other_cpty_id_country_code_gleif     AS oth_country,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector,
        -- product
        CONCAT('XCBT_FFDPSX_', to_date(e.leg1_maturity_date)) AS product_id,
        'US'                                                 AS product_country,
        to_date(e.leg1_maturity_date)                        AS expiration_date,
        -- trade terms
        e.leg1_notional_leg1           AS notional,
        e.leg1_notional_leg1_currency  AS notional_currency,
        e.leg1_price                   AS price,
        e.leg1_notional_quantity_leg1  AS quantity
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    LEFT JOIN {schema}.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN {schema}.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period BETWEEN '{start}' AND '{end}'
        AND e.leg1_execution_venue = 'XCBT'
        AND e.leg1_contract_type = 'FUTR'
        AND e.leg1_asset_class = 'INTR'
        AND e.leg1_product_cfi = 'FFDPSX'
        AND e.leg1_notional_leg1_currency = 'USD'
        AND e.leg1_maturity_date IS NOT NULL
        AND e.leg1_direction IN ('BYER', 'SLLR')
        AND e.leg1_notional_leg1 > 0
        AND e.leg1_notional_leg1 <= 1e11
) base
"""

In [ ]:
# test: one day per regime and region
tests = {
    'prerefit_ea': prerefit_query('2023-07-17', '2023-07-17'),
    'prerefit_us': prerefit_us_query('2023-07-17', '2023-07-17'),
    'refit_ea':    refit_query('2024-07-15', '2024-07-15'),
    'refit_us':    refit_us_query('2024-07-15', '2024-07-15'),
}
dfs = {k: pd.read_sql_query(q, cnxn) for k, q in tests.items()}
for k, d in dfs.items():
    print(k, d.shape, list(d.columns) == list(dfs['refit_ea'].columns))
dfs['refit_us'].head()

In [ ]:
# create pre-REFIT table (EA and US)
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut_prerefit")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut_prerefit
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{prerefit_query(prerefit_start, prerefit_end)}
UNION ALL
{prerefit_us_query(prerefit_start, prerefit_end)}
""")

In [ ]:
# create REFIT table (EA and US)
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut_refit")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut_refit
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
{refit_query(refit_start, refit_end)}
UNION ALL
{refit_us_query(refit_start, refit_end)}
""")

In [ ]:
# full series: pre-REFIT 2018-01-01 to 2024-04-26, REFIT 2024-04-29 to 2026-09-01
cursor.execute(f"DROP TABLE IF EXISTS {schema}.hermesf_fut")
cursor.execute(f"""
CREATE EXTERNAL TABLE {schema}.hermesf_fut
STORED AS PARQUET TBLPROPERTIES ('external.table.purge'='true')
AS
SELECT * FROM {schema}.hermesf_fut_prerefit
UNION ALL
SELECT * FROM {schema}.hermesf_fut_refit
""")

In [ ]:
# check
pd.read_sql_query(f"""
SELECT product_country, MIN(reference_period) AS first_date, MAX(reference_period) AS last_date,
       COUNT(*) AS n_rows, COUNT(DISTINCT product_id) AS n_contracts
FROM {schema}.hermesf_fut
GROUP BY product_country
ORDER BY product_country
""", cnxn)